In [ ]:
import sys
from pathlib import Path

# Inyectar la raíz del proyecto al sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CONFIG, get_external_bronze_dir
from src.ingestion.kaggle_downloader import download_and_organize_enemdu

print(f"✅ Proyecto cargado correctamente: {CONFIG['project']['name']}")
print(f"📁 Ruta raíz: {PROJECT_ROOT}")

In [ ]:
# La función encapsula la autenticación, descarga, extracción y clasificación en Bronze
inventario = download_and_organize_enemdu()

print("Resumen de archivos procesados:")
print(f" - Diccionarios: {len(inventario['diccionarios'])}")
print(f" - Microdatos CSV: {len(inventario['microdatos_csv'])}")

In [ ]:
import os
import zipfile
from src.ingestion.kaggle_downloader import download_and_organize_enemdu
# 1. Autenticar con Kaggle usando el token del .env
api = KaggleApi()
api.authenticate()

dataset_repo = CONFIG["sources"]["enemdu"]["kaggle_dataset"]
print(f"🚀 Descargando dataset desde Kaggle: {dataset_repo}...")

# 2. Descargar y extraer en la carpeta Bronze de ENEMDU
api.dataset_download_files(dataset_repo, path=str(enemdu_bronze_dir), unzip=True)
print("✅ Descarga y descompresión completadas.")

# 3. Separar Diccionarios (.xlsx) y Microdatos (.csv) en subcarpetas
dicts_dir = enemdu_bronze_dir / "diccionarios"
csvs_dir = enemdu_bronze_dir / "microdatos_csv"

dicts_dir.mkdir(exist_ok=True)
csvs_dir.mkdir(exist_ok=True)

for item in enemdu_bronze_dir.glob("*"):
    if item.is_file():
        if item.suffix.lower() in [".xlsx", ".xls"]:
            item.rename(dicts_dir / item.name)
        elif item.suffix.lower() == ".csv":
            item.rename(csvs_dir / item.name)

print(f"📁 Organización en Bronze terminada:")
print(f"   - Diccionarios en: {dicts_dir}")
print(f"   - CSVs por año en: {csvs_dir}")